# Downside-Beta -- a quantitative teardown 🔬
### Cross-sectional pricing · β⁻ vs β vs β⁻−β · HAC inference · block bootstrap · capacity

![Signal: Mixed](https://img.shields.io/badge/Signal-Mixed-dab617?style=flat-square)
![Tradability: Mirage](https://img.shields.io/badge/Tradability-Mirage-c0392b?style=flat-square)
![Distinct from beta%3F: Not_supported](https://img.shields.io/badge/Distinct_from_beta%3F-Not_supported-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb) -- *same seven beats, every claim now carrying its standard error.* We test Ang-Chen-Xing (2006) as a cross-sectional pricing claim and separate the raw downside-beta sort from its incremental (over-beta) content, which is what decides whether 'downside risk' is a factor or a costume.

> **Not investment advice.** Yahoo daily adjusted closes (total-return adjusted) for a large-cap S&P 500 basket, equal-weight market proxy; downside beta on a trailing 252-day window; one-month execution lag; HAC *t* (Newey-West) and circular block-bootstrap CIs. Sources: [`docs/references.md`](../docs/references.md), pinned run: [`docs/results.md`](../docs/results.md).
>
> 💡 **The `💡 In plain words` notes** translate each result back into intuition. House style in [METHODOLOGY.md](../../../METHODOLOGY.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root (quantlab/)
%matplotlib inline
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9.5, 5.0), "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
RED, AMBER, GREEN, GREY = "#c0392b", "#dab617", "#2ea44f", "#8b949e"
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

from downside_beta import data, strategy as st

# --- the real tape is cache-first; fall back to the synthetic control offline ---
HAVE_REAL = os.path.exists(data.PANEL_CACHE) and os.path.exists(data.MARKET_CACHE)
if HAVE_REAL:
    try:
        rdf_real, mkt_real = data.load_real(fetch=False)
        # drop the in-progress month
        last = rdf_real.index[-1]
        cut = pd.Timestamp(last.year, last.month, 1) - pd.Timedelta(days=1)
        rdf_real, mkt_real = rdf_real.loc[:cut], mkt_real.loc[:cut]
        print(f"REAL tape: {rdf_real.shape[0]} days x {rdf_real.shape[1]} tickers, "
              f"{rdf_real.index[0].date()} -> {rdf_real.index[-1].date()}")
    except Exception as e:
        HAVE_REAL = False
        print("Real cache unreadable, falling back to synthetic:", e)
if not HAVE_REAL:
    print("No real cache -- notebook will run on the SYNTHETIC tape and quote frozen "
          "real numbers from docs/results.md.")


REAL tape: 5384 days x 40 tickers, 2005-01-04 -> 2026-05-29


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| **Signal** | Mixed | Raw β⁻ spread +12.17%/yr, HAC *t* = +2.56; but relative β⁻−β earns -0.46%/yr, *t* = -0.14 -- real on the raw sort, none on the downside-specific part. Survivorship-biased basket. |
| **Tradability** | Mirage | Net of 10 bps one-way, *t* falls to +1.55 (below the bar), before short borrow; the tradeable part is plain beta. |
| **Distinct from plain beta?** | Not supported | β⁻−β earns nothing (*t* = -0.14); the premium is the beta premium relabelled. |

> 💡 **In plain words.** The downside-beta sort 'works' the way a thermometer in a sauna 'works' -- it reads hot, but it's measuring the room (beta), not a new thing (downside risk).

## 1 · The claim, steelmanned

Ang, Chen & Xing (2006) decompose market beta around the market mean:

$$\beta^- = \frac{\mathrm{cov}(r_i, r_m \mid r_m < \mu_m)}{\mathrm{var}(r_m \mid r_m < \mu_m)}, \qquad \beta^+ = \frac{\mathrm{cov}(r_i, r_m \mid r_m > \mu_m)}{\mathrm{var}(r_m \mid r_m > \mu_m)}.$$

- **H₁ (the headline):** sorting on β⁻ yields a positive high-minus-low spread with HAC *t* ≥ 2.
- **H₂ (the decisive one):** the spread survives controlling for plain β -- operationalised as a sort on **β⁻ − β** (their own relative downside beta) earning a positive, significant spread.
- **H₃ (tradability):** the net-of-cost spread keeps *t* ≥ 2 at realistic costs.

We use μ_m = 0 (down = a negative market day) as the canonical threshold; a market-mean threshold is available as a robustness knob.

## 2 · So what? -- what rides on each answer

H₂ is the whole game. A raw β⁻ premium that vanishes once β is removed is not a new factor; it is the CAPM market premium, harvested through an expensive long-short instead of a cheap index. Confirming H₁ while rejecting H₂ is exactly the kind of 'real but misattributed' result this desk exists to flag.

## 3 · How we'd know -- the protocol

**Decompose** → estimate (β, β⁻, β⁺) on a trailing 252-day window per name. **Sort** → monthly quintile long-short on each of β⁻, β, β⁻−β, with one execution lag (signal at *t*−1 month-end earns month *t*). **Robust inference** → Newey-West HAC *t* and circular block-bootstrap CIs. **Control** → random same-sized partitions (concentration null). **Capacity** → one-way × NAV cost sweep, both legs. **Positive control** → synthetic tape with a planted premium.

## 4 · The teardown

### 4a · Positive/null control -- the harness is a faithful detector

Before trusting any real number, prove the machine detects the effect it claims to. On a synthetic firm × day panel we plant a downside premium (or none) and check the sort recovers it (or doesn't).

In [2]:
rows = []
for prem, label in [(0.0, 'null'), (0.0025, 'priced')]:
    rdf, mkt, truth = data.synthetic_panel(downside_premium=prem, seed=332)
    res = st.monthly_quantile_returns(rdf, mkt, signal_col='beta_dn')
    s = st.summarize(res['spread'])
    lo, hi = st.block_bootstrap_ci(res['spread'], n_boot=1000)
    rows.append({'tape': label, 'spread %/yr': s['mean_ann']*100, 'HAC t': s['tstat'],
                 'CI lo bps/mo': lo*1e4, 'CI hi bps/mo': hi*1e4, 'n': s['n']})
    if prem > 0:
        panel = st.beta_panel(rdf, mkt, window_end=rdf.index[-1])
        corr = np.corrcoef(panel['rel_dbeta'].to_numpy(), truth['gamma'])[0,1]
ctrl = pd.DataFrame(rows).set_index('tape'); display(ctrl.round(2))
print(f'corr(estimated β⁻−β, planted downside loading) = {corr:.2f} '
      '-- the sort really keys on downside risk when downside risk is what varies.')

,spread %/yr,HAC t,CI lo bps/mo,CI hi bps/mo,n
tape,,,,,
null,-8.410,-0.850,-230.650,77.620,109
priced,30.020,2.870,82.650,408.440,109


corr(estimated β⁻−β, planted downside loading) = 0.92 -- the sort really keys on downside risk when downside risk is what varies.


> 💡 **In plain words.** The machine bags the planted premium (*t* ≈ +2.9) and finds nothing when there's nothing to find (*t* ≈ −0.9). It is a trustworthy detector -- so when it finds the *real*-tape premium is all-beta, we believe it. (This synthetic result is a machinery proof; it can never *back* a stamp.)

### 4b · The real tape -- β⁻ vs β vs β⁻−β

In [3]:
if HAVE_REAL:
    tape = 'REAL'
    out = {}
    for c in ['beta_dn', 'beta', 'rel_dbeta']:
        res = st.monthly_quantile_returns(rdf_real, mkt_real, signal_col=c, min_stocks=8)
        s = st.summarize(res['spread'])
        lo, hi = st.block_bootstrap_ci(res['spread'], n_boot=1000)
        out[c] = {'spread %/yr': s['mean_ann']*100, 'HAC t': s['tstat'],
                  'ann SR': s['sharpe_ann'], 'hit': s['hit_rate'],
                  'CI lo bps/mo': lo*1e4, 'CI hi bps/mo': hi*1e4, 'n': s['n']}
    real_tbl = pd.DataFrame(out).T
else:
    tape = 'FROZEN (docs/results.md)'
    real_tbl = pd.DataFrame({
        'beta_dn':   {'spread %/yr': 12.17, 'HAC t': 2.56, 'ann SR': 0.59, 'hit': 0.59, 'CI lo bps/mo': 19, 'CI hi bps/mo': 176, 'n': 250},
        'beta':      {'spread %/yr': 10.46, 'HAC t': 2.09, 'ann SR': 0.48, 'hit': 0.56, 'CI lo bps/mo': np.nan, 'CI hi bps/mo': np.nan, 'n': 250},
        'rel_dbeta': {'spread %/yr': -0.46, 'HAC t': -0.14, 'ann SR': -0.03, 'hit': 0.48, 'CI lo bps/mo': -58, 'CI hi bps/mo': 50, 'n': 250},
    }).T
print(f'[{tape}] monthly long-short spreads, gross:')
display(real_tbl.round(2))

[REAL] monthly long-short spreads, gross:


,spread %/yr,HAC t,ann SR,hit,CI lo bps/mo,CI hi bps/mo,n
beta_dn,12.170,2.560,0.590,0.590,22.260,170.950,250.000
beta,10.460,2.090,0.480,0.560,7.020,165.290,250.000
rel_dbeta,-0.460,-0.140,-0.030,0.480,-59.720,51.250,250.000


> 💡 **In plain words.** Read the three rows top to bottom: β⁻ clears the bar, β clears it too at nearly the same size, and β⁻−β (the only one that would make 'downside' a distinct factor) is a flat line through zero. H₁ holds; H₂ fails.

## 5 · The verdict -- the decisive statistics in one place

- **Signal MIXED.** β⁻: +12.17%/yr, HAC *t* = +2.56, 95% block-boot CI [+19, +176] bps/mo (excludes 0). β⁻−β: -0.46%/yr, *t* = -0.14, CI [-58, +50] bps/mo (straddles 0). Real on the raw sort · None on the downside-specific part.
- **Distinct from plain beta? NOT SUPPORTED.** β sort earns +10.46%/yr (*t* = +2.09); removing β removes the whole premium.
- **Tradability MIRAGE.** Net *t*: +2.56 (0bps) → +2.06 (5) → +1.55 (10) → +0.54 (20), before short borrow.

Survivorship: the basket is current S&P 500 names projected backwards (opt-in guard), so even the raw β⁻ leg is an upper bound.

## 6 · Could you trade it? -- cost sweep, both legs, one-way × NAV

In [4]:
costs = [0, 5, 10, 20]
if HAVE_REAL:
    res = st.monthly_quantile_returns(rdf_real, mkt_real, signal_col='beta_dn', min_stocks=8)
    net_ann = [st.summarize(st.net_spread(res, one_way_bps=c))['mean_ann']*100 for c in costs]
    net_t   = [st.summarize(st.net_spread(res, one_way_bps=c))['tstat'] for c in costs]
    tape = 'REAL'
else:
    net_ann = [12.17, 9.77, 7.37, 2.57]
    net_t   = [2.56, 2.06, 1.55, 0.54]
    tape = 'FROZEN'
sweep = pd.DataFrame({'one-way bps': costs, 'net %/yr': net_ann, 'HAC t': net_t}).set_index('one-way bps')
print(f'[{tape}] cost sweep on the β⁻ long-short:'); display(sweep.round(2))
print('Break-even on significance lands between 5 and 10 bps -- before any short borrow.')

[REAL] cost sweep on the β⁻ long-short:


,net %/yr,HAC t
one-way bps,,
0,12.170,2.560
5,9.770,2.060
10,7.370,1.550
20,2.570,0.540


Break-even on significance lands between 5 and 10 bps -- before any short borrow.


> 💡 **In plain words.** A monthly beta-tilted long-short is expensive to run, and the only thing it harvests is the market premium -- which a buy-and-hold index fund delivers at a few basis points a year. You'd be paying a steakhouse markup for a supermarket steak.

## 7 · Going further

- **Full CRSP / survivorship-free panel** -- does a small incremental premium survive on the wider, fairer cross-section the original used?
- **Coskewness (Harvey-Siddique 2000)** as a competing measure of crash-comovement, run head-to-head with β⁻−β.
- **Threshold sensitivity** -- μ_m = market mean vs 0; conditional (downside in *bad* states only) vs unconditional downside beta.
- **The BAB mirror** -- race against Study 238 (low beta pays). PRs welcome: fork `downside_beta/strategy.py`, swap the signal column, re-run the gauntlet.